In [ ]:
  # Ver situação da GPU alocada
!nvidia-smi -L

In [ ]:
# Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuração dos paths
# path = ' /screws'
# path = '/content/drive/MyDrive/JKU/objectDetection/bikeParts'
path = '/content/drive/MyDrive/JKU/objectDetection/bikeParts/datasets/camera on the chair'
# dataset_path = '/content/drive/MyDrive/JKU/objectDetection/bikeParts/datasets'
dataset_path = '/content/drive/MyDrive/JKU/objectDetection/bikeParts/datasets/camera on the chair'

yolo_path = path + '/yolov5'
print (yolo_path)

# Área PERIGOSA

In [ ]:
# SOMENTE UMA VEZ

# Clonar yolov5
%cd '{path}'
!git clone https://github.com/ultralytics/yolov5

# !git clone --branch fix-exceeds-2gb https://github.com/zldrobit/yolov5
# !git clone https://github.com/LongxingTan/Yolov5.git

# Instalação do YOLOv5

In [ ]:
# Instalar dependências do yolov5
%cd '{yolo_path}'
!pip install -U -r requirements.txt

In [ ]:
!pip install albumentations==1.0.3

In [ ]:
# %cd '{yolo_path}'
# import torch
# import utils
# display = utils.notebook_init()  # checks
import torch
print('torch %s %s' % (torch.__version__, torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU'))

# Treinamento

In [ ]:
# Executar tensorboard nos dados de treinamento (optional)
%load_ext tensorboard
%tensorboard --logdir '{yolo_path}/runs/train'

In [ ]:
# Definição dos parâmetros do teste
model_weights = 'yolov5l6'
model_batch = 10
# images_resolution = 1024
images_resolution = 640
epochs = 300
patience = 100
conf_threshold = 0.25

train_name = f'{model_weights}-{epochs}epochs-batch{model_batch}-resolution{images_resolution}-manual_augmentation'
print(train_name)

In [ ]:
# Rodar treinamento
%cd '{yolo_path}'

# !python train.py --name '{train_name}' --resume --cache
# !python train.py --img '{images_resolution}' --batch '{model_batch}' --epochs '{epochs}' --weights '{model_weights}.pt' --data '../datasets/train.yaml' --name '{train_name}' --patience '{patience}'  --cache --exist-ok
!python train.py --img '{images_resolution}' --batch '{model_batch}' --epochs '{epochs}' --weights '{model_weights}.pt' --data '../train.yaml' --name '{train_name}' --patience '{patience}'  --cache --exist-ok


# Inferência

In [ ]:
# Inferência no conjunto de testes
%cd '{yolo_path}'
!python detect.py --line-thickness 2 --weights 'runs/train/{train_name}/weights/best.pt' --img {images_resolution} --source ../datasets/test/images --conf {conf_threshold} --name '{train_name}_conf{conf_threshold}'  --save-txt --save-conf

# Métricas

In [ ]:
# Métricas no conjunto de testes
%cd '{yolo_path}'
!python val.py --weights 'runs/train/{train_name}/weights/best.pt' --img '{images_resolution}' --name '{train_name}' --data ../datasets/test.yaml

# Exportar

In [ ]:
# Exportar
%cd '{yolo_path}'
!python export.py --weights 'runs/train/{train_name}/weights/best.pt' --include saved_model --img '{images_resolution}' --nms

# Enviar email no fim da execução

In [ ]:
import smtplib, ssl

port = 465  # For SSL
smtp_server = "smtp.gmail.com"
sender_email = "michelscheeren@alunos.utfpr.edu.br"
receiver_email = "michelhanzenscheeren@gmail.com"
password = "pgunxqslliojypti"
message = f"""From: From <michelscheeren@alunos.utfpr.edu.br>
To: To <michelhanzenscheeren@gmail.com>
Subject: Colab end

Your code execution was finished!
Name: {train_name}

"""

try:
  context = ssl.create_default_context()
  with smtplib.SMTP_SSL(smtp_server, port, context=context) as server:
      server.login(sender_email, password)
      server.sendmail(sender_email, receiver_email, message)
except Exception as error:
  print('Something went wrong!')
  print(error)

# Disconnect environment

In [ ]:
!pkill --oldest